# تست کامل خط لوله — تهران بورس (algotik-tse)

اجرای کل خط لوله: دیتابیس → دریافت داده → پاکسازی/فیچر → برچسب → دنباله → آموزش →
ارزیابی → بک‌تست → پیش‌بینی بلادرنگ.

هر سلول، بخشی از خط لوله را تست می‌کند (مجموعاً ۱۳ سلول کد).

In [1]:
# سلول ۱ — راه‌اندازی محیط
import os
import sys

PROJECT_ROOT = r"C:\Users\programmer\Desktop\trade"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.config import load_config, get_db_path, get_target_symbols, setup_logging

setup_logging(force=True)
CONFIG = load_config()
DB_PATH = get_db_path()
TARGET_SYMBOLS = get_target_symbols()

print("Project :", CONFIG.get("project"))
print("DB      :", DB_PATH)
print("Targets :", len(TARGET_SYMBOLS), TARGET_SYMBOLS)

import algotik_tse as att
import torch

print("algotik-tse :", getattr(att, "__version__", "?"))
print("torch       :", torch.__version__)
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)

Project : {'name': 'Bourse Predictor', 'version': '0.1.0'}
DB      : C:\Users\programmer\Desktop\trade\data\bourse.db
Targets : 18 ['فولاد', 'فخوز', 'ذوب', 'شپنا', 'شتران', 'تاپیکو', 'وبملت', 'وتجارت', 'وبصادر', 'خودرو', 'خساپا', 'دارو', 'تیپیکو', 'همراه', 'اخابر', 'وپارس', 'شستا', 'غبشهر']


algotik-tse : 1.8.0
torch       : 2.14.0+cpu
pandas      : 2.3.3
numpy       : 2.3.5


In [2]:
# سلول ۲ — سلامت اسکیمای دیتابیس
from src import db as dbm
from src.database.schema import list_tables

dbm.ensure_schema(DB_PATH)
tables = list_tables(DB_PATH)

conn = dbm.connect(DB_PATH)
for table in tables:
    print(f"{table:20s} → {dbm.table_count(conn, table):>10,} ردیف")
conn.close()

assert len(tables) >= 5, "جدول‌های پایه دیتابیس وجود ندارند"
print("✅ اسکیما سالم است:", tables)

15:48:44 | SUCCESS  | ✅ دیتابیس ساخته شد: C:\Users\programmer\Desktop\trade\data\bourse.db | 24 دستور اجرا شد


15:48:44 | INFO     | اسکیمای دیتابیس با create_database() آماده شد


client_type          →      1,647 ردیف
codal_news           →          0 ردیف
daily_history        →      1,643 ردیف
index_companies      →          0 ردیف
indices              →          0 ردیف
instruments          →        428 ردیف
intraday_candles     →          0 ردیف
market_snapshot      →          0 ردیف
order_book           →          0 ردیف
queue                →          0 ردیف
trades               →          0 ردیف
✅ اسکیما سالم است: ['client_type', 'codal_news', 'daily_history', 'index_companies', 'indices', 'instruments', 'intraday_candles', 'market_snapshot', 'order_book', 'queue', 'trades']


In [3]:
# سلول ۳ — سلامت کلاینت TSETMC و اعتبارسنجی نمادهای هدف
from src.data_ingestion.tsetmc_client import TsetmcClient

client = TsetmcClient(retries=2, retry_delay=0.5, request_delay=0.2)
symbols_df = client.filter_stocks(client.get_symbols())
print(f"ابزار stock: {len(symbols_df)}")

check = client.validate_symbols(TARGET_SYMBOLS, symbols_df)
print("AVAILABLE:", check["available"])
print("MISSING  :", check["missing"])
assert not check["missing"], f"نمادهای گمشده: {check['missing']}"
assert "شستا" in check["available"], "نماد جایگزین «شستا» باید موجود باشد"
print("✅ همه نمادهای هدف در TSETMC موجودند")

Getting all of bourse symbols...


The dataframe of all bourse symbols is ready! (428 symbols)


15:48:57 | INFO     | تعداد 428 ابزار دریافت شد


15:48:57 | INFO     | 18 نماد از 18 نماد هدف تأیید شد


ابزار stock: 428
AVAILABLE: ['فولاد', 'فخوز', 'ذوب', 'شپنا', 'شتران', 'تاپیکو', 'وبملت', 'وتجارت', 'وبصادر', 'خودرو', 'خساپا', 'دارو', 'تیپیکو', 'همراه', 'اخابر', 'وپارس', 'شستا', 'غبشهر']
MISSING  : []
✅ همه نمادهای هدف در TSETMC موجودند


In [4]:
# سلول ۴ — همگام‌سازی جدول instruments از TSETMC
from src.data_ingestion.backfill import BackfillRunner

market_conn = dbm.connect(DB_PATH)
runner = BackfillRunner(client, market_conn, "2024-01-01", "2026-12-31")
stocks = runner.fetch_symbols(CONFIG.get("markets") or {"bourse": True})

n = runner.run_instruments(stocks)
cur = market_conn.execute("SELECT COUNT(*) FROM instruments").fetchone()[0]
print(f"upsert instruments: {n} | مجموع در جدول: {cur}")
assert cur >= 400, "جدول instruments باید حداقل ۴۰۰ ردیف داشته باشد"
print("✅ instruments همگام شد")

Getting all of bourse symbols...


The dataframe of all bourse symbols is ready! (428 symbols)


15:49:10 | INFO     | تعداد 428 ابزار دریافت شد


15:49:10 | INFO     | 428 ابزار از نوع stock از 428 ابزار


15:49:10 | SUCCESS  | instruments: 428 سطر upsert شد


upsert instruments: 428 | مجموع در جدول: 428
✅ instruments همگام شد


In [5]:
# سلول ۵ — Backfill تاریخچه + حقیقی/حقوقی (نمادهای تست)
from datetime import date, timedelta

TEST_SYMBOLS = ["فولاد", "شپنا", "شستا"]
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=400)   # تقریباً یک سال معاملاتی

targets = client.validate_symbols(TEST_SYMBOLS, stocks)["frame"]
print("نمادهای هدف تست:", list(targets["symbol"]))

n_h = runner.run_daily_history(targets)
n_c = runner.run_client_type(targets)
runner.report()
print(f"history: {n_h} | client_type: {n_c}")
assert n_h > 0 and n_c > 0, "Backfill تاریخچه/حقیقی‌حقوقی باید داده تولید کند"
print("✅ Backfill انجام شد")

15:49:10 | INFO     | 3 نماد از 3 نماد هدف تأیید شد


15:49:10 | INFO     | [1/3] تاریخچه «فولاد»


نمادهای هدف تست: ['فولاد', 'شپنا', 'شستا']


15:49:12 | INFO     | «فولاد»: 506 سطر (20240101 تا 20260913)


15:49:12 | INFO     | [2/3] تاریخچه «شپنا»


15:49:13 | INFO     | «شپنا»: 560 سطر (20240101 تا 20260913)


... (11 outputs left out)

15:49:17 | INFO     | instruments    :      428 سطر


15:49:17 | INFO     | daily_history  :     1643 سطر


15:49:17 | INFO     | client_type    :     1647 سطر


history: 1643 | client_type: 1647
✅ Backfill انجام شد


In [6]:
# سلول ۶ — بررسی کیفیت داده‌ها در دیتابیس
hist = dbm.load_daily_history(symbols=TEST_SYMBOLS, db_path=DB_PATH)
clt = dbm.load_client_type(symbols=TEST_SYMBOLS, db_path=DB_PATH)
market_conn.close()

print("history:", hist.shape, "| symbols:", hist["symbol"].nunique())
print(hist.groupby("symbol")["gregorian_date"].agg(["count", "min", "max"]))
print()
print("client :", clt.shape, "| columns:", len(clt.columns))
nan = hist[["open", "high", "low", "close", "volume"]].isna().mean()
print("NaN درصدی ستون‌های اصلی:\n", nan.round(4))

assert len(hist) >= 180, "برای ساخت دنباله با seq_len=60 به داده بیشتری نیاز است"
assert hist["symbol"].nunique() >= 3, "هر سه نماد تست باید تاریخچه داشته باشند"
print("✅ داده به اندازه کافی و با کیفیت است")

history: (1643, 13) | symbols: 3
        count       min       max
symbol                           
شستا      577  20240101  20260913
شپنا      560  20240101  20260913
فولاد     506  20240101  20260913

client : (1647, 24) | columns: 24
NaN درصدی ستون‌های اصلی:
 open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64
✅ داده به اندازه کافی و با کیفیت است


In [7]:
# سلول ۷ — جدول فیچر (پاکسازی + تکنیکال + فصلی + جریان پول)
from src.training.dataset import load_feature_table

table = load_feature_table(symbols=TEST_SYMBOLS, db_path=DB_PATH)
print("feature table:", table.shape)
print("columns:", list(table.columns))
assert len(table) > 0 and len(table.columns) > 20
print("✅ جدول فیچر ساخته شد")

15:49:18 | INFO     | clean_history: 1643 → 1591 سطر


15:49:18 | INFO     | فیچرهای تکنیکال ساخته شد (29 ستون جدید)


15:49:18 | INFO     | فیچرهای تقویمی اضافه شد (1591 سطر)


15:49:21 | INFO     | clean_client_type: 1647 → 1647 سطر


15:49:21 | INFO     | فیچرهای جریان پول ساخته شد (1647 سطر)


15:49:21 | WARNING  | خبری برای ساخت فیچر احساسات وجود ندارد


15:49:21 | WARNING  | جدول order_book خالی است


15:49:21 | INFO     | جدول فیچر: 1591 سطر × 85 ستون


feature table: (1591, 85)
columns: ['symbol', 'ins_code', 'gregorian_date', 'jalali_date', 'open', 'high', 'low', 'close', 'last', 'volume', 'value', 'trade_count', 'date', 'sma_5', 'close_to_sma_5', 'sma_10', 'close_to_sma_10', 'sma_20', 'close_to_sma_20', 'sma_50', 'close_to_sma_50', 'ema_12', 'ema_26', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct', 'atr', 'atr_pct', 'stoch_k', 'stoch_d', 'return_1d', 'return_5d', 'return_20d', 'volatility_20d', 'volume_ratio_20d', 'hl_range_pct', 'jalali_year', 'jalali_month', 'jalali_day', 'jalali_season', 'week_of_month', 'day_of_week', 'is_month_end', 'is_year_end', 'hijri_month', 'is_ramadan', 'is_muharram', 'month_sin', 'month_cos', 'seasonal_trend', 'seasonal_seasonal', 'seasonal_resid', 'power_retail', 'power_institutional', 'retail_buy_share', 'retail_sell_share', 'net_val_retail', 'net_val_institutional', 'net_vol_retail', 'net_vol_institutional', 'net_val_retail_ratio', 'net_val_inst_rati

In [8]:
# سلول ۸ — برچسب‌گذاری (بالا/پایین/خنثی)
from src.training.dataset import add_labels

labeled = add_labels(table)
dist = labeled["target"].value_counts().sort_index()
print(dist)
print(f"rows: {len(labeled)} | horizon_days: {labeled['future_return'].notna().sum()} با آینده")
assert labeled["target"].nunique() >= 2, "حداقل دو کلاس باید وجود داشته باشد"
print("✅ برچسب‌ها ساخته شد")

15:49:21 | INFO     | برچسب‌ها (افق 1 روز، آستانه 1.0%): down=29.5% neutral=34.6% up=35.9%


target
0    469
1    549
2    570
Name: count, dtype: int64
rows: 1588 | horizon_days: 1588 با آینده
✅ برچسب‌ها ساخته شد


In [9]:
# سلول ۹ — دنباله‌های زمانی + بازه‌های Walk-Forward
from src.training.dataset import build_sequences, walk_forward_splits

data = build_sequences(labeled)
X, y = data["X"], data["y"]
print("X:", X.shape, "| y:", y.shape)
print("نمادها:", np.unique(data["symbols"]))
print("تعداد فیچر:", len(data["feature_cols"]))

folds = walk_forward_splits(data["dates"], n_splits=3)
for i, (tr, va) in enumerate(folds, start=1):
    print(f"fold {i}: train={len(tr)} val={len(va)}")
assert X.size > 0 and len(folds) >= 2
print("✅ دنباله‌ها و بازه‌ها آماده‌اند")

15:49:21 | INFO     | دنباله‌ها: X=(1408, 60, 80) y=(1408,)


15:49:21 | INFO     | 3 بازه Walk-Forward ساخته شد


X: (1408, 60, 80) | y: (1408,)
نمادها: ['شستا' 'شپنا' 'فولاد']
تعداد فیچر: 80
fold 1: train=352 val=352
fold 2: train=704 val=352
fold 3: train=1056 val=352
✅ دنباله‌ها و بازه‌ها آماده‌اند


In [10]:
# سلول ۱۰ — آموزش مدل (smoke test با epoch کم)
from src.training.train import train

summary = train(model_type="lstm", symbols=TEST_SYMBOLS, epochs=5, splits=3)
print("summary:", summary)
assert summary.get("folds", 0) > 0
print("✅ آموزش مدل کار کرد")

15:49:22 | INFO     | clean_history: 1643 → 1591 سطر


15:49:22 | INFO     | فیچرهای تکنیکال ساخته شد (29 ستون جدید)


15:49:22 | INFO     | فیچرهای تقویمی اضافه شد (1591 سطر)


15:49:22 | INFO     | clean_client_type: 1647 → 1647 سطر


15:49:22 | INFO     | فیچرهای جریان پول ساخته شد (1647 سطر)


15:49:22 | WARNING  | خبری برای ساخت فیچر احساسات وجود ندارد


... (22 outputs left out)

15:50:22 | SUCCESS  | گزارش ارزیابی ذخیره شد: C:\Users\programmer\Desktop\trade\logs\evaluation\lstm_fold3_20260914_155022.json


15:50:22 | SUCCESS  | لاگ اجرا ذخیره شد: C:\Users\programmer\Desktop\trade\models\saved\runs\train_lstm_20260914_154922.json


15:50:22 | SUCCESS  | میانگین دقت جهت‌دهی روی 3 بازه: 35.30%


summary: {'folds': 3, 'accuracy_mean': 0.37973484848484845, 'accuracy_std': 0.09383127025314451, 'directional_accuracy_mean': 0.35304139320867806, 'directional_accuracy_std': 0.24067638246681053, 'f1_macro_mean': 0.2984822162831659, 'f1_macro_std': 0.10205082043031756, 'auc_roc_mean': 0.5860919952058999, 'auc_roc_std': 0.04570516195463799}
✅ آموزش مدل کار کرد


In [11]:
# سلول ۱۱ — تولید سیگنال‌های معاملاتی (Walk-Forward درجا برای بک‌تست)
from src.config import get_section
from src.models.lstm_model import build_lstm
from src.training.dataset import standardize
from src.training.train import _train_torch_fold, set_seed

set_seed(42)
train_cfg = dict(get_section("model", {}).get("training", {}))
train_cfg.update({"epochs": 6, "early_stopping_patience": 4})

closes = labeled.set_index(["symbol", "gregorian_date"])["close"].to_dict()
sig_map = {0: "down", 1: "neutral", 2: "up"}
signals = []
for idx, (tr_idx, va_idx) in enumerate(folds, start=1):
    tx, vx = standardize(data["X"][tr_idx], data["X"][va_idx])
    ty, vy = data["y"][tr_idx], data["y"][va_idx]
    model = build_lstm(tx.shape[-1], get_section("model", {}).get("lstm", {}))
    res = _train_torch_fold(model, tx, ty, vx, vy, train_cfg)
    for j, gi in enumerate(va_idx):
        sym, d = data["symbols"][gi], int(data["dates"][gi])
        signals.append({
            "symbol": sym,
            "gregorian_date": d,
            "close": closes[(sym, d)],
            "signal": sig_map[int(res["y_proba"][j].argmax())],
        })

sig_df = pd.DataFrame(signals)
print("signals:", sig_df.shape)
print(sig_df["signal"].value_counts().to_dict())
assert not sig_df.empty
print("✅ سیگنال‌های تست برای بک‌تست تولید شد")

15:50:24 | INFO     | epoch   1 | train_loss=1.0742 | val_loss=1.2786


15:50:32 | INFO     | epoch   5 | train_loss=0.8977 | val_loss=1.3454


15:50:34 | INFO     | Early stopping در epoch 6


15:50:38 | INFO     | epoch   1 | train_loss=1.0973 | val_loss=1.1315


15:50:51 | INFO     | epoch   5 | train_loss=0.9302 | val_loss=1.2162


15:50:54 | INFO     | Early stopping در epoch 6


15:51:00 | INFO     | epoch   1 | train_loss=1.0840 | val_loss=1.0407


15:51:20 | INFO     | epoch   5 | train_loss=0.9422 | val_loss=1.0394


15:51:24 | INFO     | Early stopping در epoch 6


signals: (1056, 4)
{'neutral': 524, 'up': 413, 'down': 119}
✅ سیگنال‌های تست برای بک‌تست تولید شد


In [12]:
# سلول ۱۲ — بک‌تست با شبیه‌ساز
from src.backtesting.simulator import Simulator

sim = Simulator()
result = sim.run(sig_df)
print(result.metrics)
assert result.metrics.get("trades", -1) >= 0
report_path = sim.report_html(result, name="pipeline_test")
print("گزارش:", report_path)
print("✅ شبیه‌ساز کار کرد")

15:51:25 | SUCCESS  | بک‌تست: بازدهی 72.75% | Sharpe 1.41 | MaxDD -33.77% | WinRate 88.9% | 18 معامله


{'total_return': 0.7275220885296247, 'final_equity': 172752208.85296246, 'sharpe': 1.4139510872413525, 'max_drawdown': -0.337656247526983, 'win_rate': 0.8888888888888888, 'trades': 18, 'avg_trade_return': 0.10693159961045098}


15:51:26 | SUCCESS  | گزارش بک‌تست: C:\Users\programmer\Desktop\trade\logs\evaluation\pipeline_test_20260914_155125.html


گزارش: C:\Users\programmer\Desktop\trade\logs\evaluation\pipeline_test_20260914_155125.html
✅ شبیه‌ساز کار کرد


In [13]:
# سلول ۱۳ — پیش‌بینی آخرین روز (خط لوله بلادرنگ)
from src.realtime.pipeline import Predictor

predictor = Predictor(model_type="lstm")
latest = predictor.predict_latest(TEST_SYMBOLS)
print(latest)
assert not latest.empty
print()
print("🎉 همه ۱۳ سلول خط لوله با موفقیت اجرا شد")

15:51:26 | INFO     | clean_history: 1643 → 1591 سطر


15:51:26 | INFO     | فیچرهای تکنیکال ساخته شد (29 ستون جدید)


15:51:26 | INFO     | فیچرهای تقویمی اضافه شد (1591 سطر)


15:51:27 | INFO     | clean_client_type: 1647 → 1647 سطر


15:51:27 | INFO     | فیچرهای جریان پول ساخته شد (1647 سطر)


15:51:27 | WARNING  | خبری برای ساخت فیچر احساسات وجود ندارد


... (3 outputs left out)

15:51:27 | INFO     | دنباله‌ها: X=(1408, 60, 80) y=(1408,)


15:51:27 | INFO     | مدل بارگذاری شد: C:\Users\programmer\Desktop\trade\models\saved\lstm_fold3.pt


15:51:27 | INFO     | سیگنال برای 3 نماد تولید شد


  symbol  gregorian_date signal  probability  expected_return   close  \
0   شستا        20260905     up       0.3748          -0.3750  2920.0   
1   شپنا        20260622   down       0.4396           0.1008  8688.0   
2  فولاد        20260912     up       0.6476           0.5218  2967.0   

   horizon_days  
0             1  
1             1  
2             1  

🎉 همه ۱۳ سلول خط لوله با موفقیت اجرا شد
